## Шаг 6. Постановка и решение оптимизационной задачи



### 6.1 Постановка оптимизационной задачи (4 критерия)



**Сутевое описание проблемы:**
В условиях выхода из глобального кризиса (переход от графа локдауна $G_3$ к стадии восстановления $G_4$) правительства и авиационные альянсы сталкиваются с проблемой ограниченных ресурсов. Восстановить все 100% рейсов мгновенно невозможно. 

Возникает задача проектирования **«Оптимальной опорной сети» (Optimal Backbone Network)** — необходимо выбрать такой подграф (набор рейсов), который свяжет ключевые регионы мира, минимизируя издержки, но при этом сохраняя устойчивость к повторным шокам. 

Если мы будем искать только математическое *остовное дерево минимального веса (MST)*, мы получим самую дешевую сеть, но она будет обладать нулевой живучестью: отмена любого одного рейса приведет к изоляции целых континентов. Поэтому задача требует многокритериальной балансировки.

**4 критерия оптимизации:**

1. **Минимизация операционных затрат (Экономика):** 
   Минимизация суммы длин всех выбранных маршрутов (весов ребер `distance`). Отражает расходы на топливо и амортизацию бортов.
   * $f_1 \rightarrow \min$

2. **Максимизация покрытия пассажиропотока (Социальная функция):** 
   Максимизация суммарного веса ребер `passengers_per_year`. Сеть должна в первую очередь удовлетворять глобальный спрос.
   * $f_2 \rightarrow \max$

3. **Максимизация стратегической устойчивости (Интеграция ATI):** 
   Максимизация суммарного топологического индекса $ATI$ (из Шага 5) узлов, которые покрываются выбранными рейсами. Приоритет отдается связям между "Супер-хабами", так как они генерируют максимальную связность.
   * $f_3 \rightarrow \max$

4. **Обеспечение топологической живучести (Резервирование/Resilience):** 
   Максимизация коэффициента кластеризации (или поддержание минимальной степени вершин $degree \ge 2$). В отличие от остовного дерева, наша сеть обязана иметь циклические маршруты-дублеры, чтобы закрытие одного хаба не парализовало систему.
   * $f_4 \rightarrow \max$

**Ограничение (Constraint):** Граф выбранной опорной сети должен оставаться **связным** (одна компонента связности для всех выбранных узлов).

## 6.2. Формализованная запись оптимизационной задачи



Введем неориентированный взвешенный граф восстанавливающейся сети $G = (V, E)$, где $V$ — множество аэропортов, а $E$ — множество возможных маршрутов.

**Область допустимых значений (Переменная принятия решений):**
Определим булеву переменную (тумблер) для каждого ребра $(i,j) \in E$:
$$x_{ij} = \begin{cases} 1, & \text{если маршрут между аэропортами } i \text{ и } j \text{ сохраняется} \\ 0, & \text{иначе} \end{cases}$$

Следовательно, область допустимых значений $X = \{0, 1\}^{|E|}$.

**Описание параметров целевой функции:**
Для каждого ребра $(i,j) \in E$ и узла $i \in V$ заданы известные константы:
*   $d_{ij}$ — расстояние между аэропортами $i$ и $j$ (затраты/Cost).
*   $p_{ij}$ — годовой пассажиропоток на маршруте $i \leftrightarrow j$ (Flow).
*   $ATI_i, ATI_j$ — агрегированный топологический индекс важности узлов (Стратегическая ценность из шага 5.4).
*   $D_{max}$ — максимально допустимый бюджет суммарной протяженности сети (ограничение ресурсов).
*   $\alpha_1, \alpha_2, \alpha_3, \alpha_4$ — весовые коэффициенты (priorities) для каждого из критериев, где $\sum \alpha_k = 1$.

**Целевая функция на экстремум (Максимизация совокупной полезности):**
Поскольку задача является многокритериальной, применим метод линейной свертки (Weighted Sum Model). Мы переводим минимизацию затрат в максимизацию (со знаком минус) и объединяем 4 критерия из п. 6.1 в единый функционал (Fitness Function):

$$ F(x) = \sum_{(i,j) \in E} x_{ij} \cdot \Big[ \alpha_1 \tilde{p}_{ij} + \alpha_2 (\widetilde{ATI}_i + \widetilde{ATI}_j) + \alpha_3 \tilde{r}_{ij} - \alpha_4 \tilde{d}_{ij} \Big] \longrightarrow \max $$

*Где знак тильды ($\sim$) означает предварительную Min-Max нормализацию параметров на отрезок $[0, 1]$, чтобы складывать "яблоки с яблоками". Параметр $\tilde{r}_{ij} \equiv 1$ символизирует бонус за добавление каждого избыточного ребра для повышения топологической живучести.*

**Система ограничений:**
Оптимизация проводится при соблюдении следующих строгих условий:

1. **Ограничение бюджета на длину сети (Budget Constraint):**
   Суммарная длина всех выбранных рейсов не должна превышать ресурсный лимит $D_{max}$:
   $$ \sum_{(i,j) \in E} x_{ij} \cdot d_{ij} \le D_{max} $$

2. **Гарантия связности (Connectivity Constraint):**
   Для любого разбиения вершин выбранного подграфа на два непересекающихся множества $S$ и $V \setminus S$, должно существовать хотя бы одно ребро, соединяющее их (чтобы сеть не распалась на острова):
   $$ \sum_{i \in S, j \notin S} x_{ij} \ge 1, \quad \forall S \subset V, S \neq \emptyset $$

3. **Ограничение на включение Супер-Хабов (Strategic Hubs Inclusion):**
   Каждый стратегически важный аэропорт (где $ATI > threshold$) обязан иметь как минимум $k$ связей (степень вершины $\ge k$) для страховки:
   $$ \sum_{j \in V} x_{ij} \ge 2, \quad \forall i \in V_{super\_hubs} $$